In [118]:
from dotenv import load_dotenv
import os
import clickhouse_connect
import pandas as pd


load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST'),
    port=int(os.getenv('CLICKHOUSE_PORT', 8123)), 
    username=os.getenv('CLICKHOUSE_USERNAME'),
    password=os.getenv('CLICKHOUSE_PASSWORD'),
    database=os.getenv('CLICKHOUSE_DATABASE')
)

In [119]:
supplier_ranking_query = """SELECT
    v.vendor_name,
    
    -- Total quantity received
    SUM(CAST(ifNull(bni.quantity_in, 0) AS Float64)) AS total_quantity_in,

    -- Avg date differences
    ROUND(
        AVG(
            dateDiff(
                'day',
                parseDateTimeBestEffortOrNull(b.shipped_date),
                parseDateTimeBestEffortOrNull(b.receipt_date)
            )
        ), 2
    ) AS avg_shipped_receipt_diff_days,
    
    ROUND(
        AVG(
            dateDiff(
                'day',
                parseDateTimeBestEffortOrNull(b.shipped_date),
                parseDateTimeBestEffortOrNull(b.bill_date)
            )
        ), 2
    ) AS avg_shipped_bill_diff_days,

    -- Vendor relationship duration
    dateDiff('month', MIN(b.created_time), today()) AS using_goods_since_months,

    -- Net monetary transaction
    ROUND(
        SUM(
            CAST(
                replaceAll(replaceAll(b.total_bcy, 'USD ', ''), ',', '') AS Float64
            )
            -
            CAST(
                replaceAll(replaceAll(ifNull(b.discount_amount_bcy, '0'), 'USD ', ''), ',', '') AS Float64
            )
        ), 2
    ) AS net_monetary_transaction,

    -- On-time delivery rate
    ROUND(
        (countIf(
            parseDateTimeBestEffortOrNull(b.receipt_date) <= parseDateTimeBestEffortOrNull(b.eta)
        ) * 100.0)
        / nullIf(countIf(b.receipt_date != '' AND b.eta != ''), 0),
        2
    ) AS on_time_delivery_rate_percent,

    -- Distinct products
    COUNT(DISTINCT bi.product_id) AS distinct_items_supplied,
COUNT(DISTINCT b.bill_id) AS total_shipments
FROM 
    zoho_books_analytics.batch_number_in AS bni
JOIN 
    zoho_books_analytics.bills AS b 
    ON bni.bill_id = b.bill_id
JOIN 
    zoho_books_analytics.bill_item AS bi 
    ON bi.bill_id = b.bill_id
JOIN 
    zoho_books_analytics.purchase_orders AS po  
    ON po.purchase_order_number = b.purchase_order
JOIN 
    zoho_books_analytics.items AS i 
    ON i.item_id = bi.product_id
JOIN 
    zoho_books_analytics.customer_item_mapping AS cim 
    ON i.sku = cim.az_sku
JOIN 
    zoho_books_analytics.vendors AS v 
    ON v.vendor_id = b.vendor_id 
WHERE 
    cim.customer_name LIKE 'Walmart%'
    AND po.po_commited != 'Direct Sale'
    AND b.receipt_date != '' 
    AND b.eta != ''
GROUP BY 
    v.vendor_name
ORDER BY 
    total_quantity_in DESC;"""

In [120]:
result = client.query(query=supplier_ranking_query)

supplier_dataset_df = pd.DataFrame(result.result_rows, columns=[col for col in result.column_names])

In [121]:
supplier_dataset_df.fillna(100, inplace=True)

In [122]:
weights = {
    "total_quantity_in": 0.13,
    "avg_shipped_receipt_diff_days": 0.13,
    "avg_shipped_bill_diff_days": 0.08,
    "using_goods_since_months": 0.09,
    "net_monetary_transaction": 0.13,
    "on_time_delivery_rate_percent": 0.22,
    "distinct_items_supplied": 0.09,
    "total_shipments": 0.13
}

In [123]:
supplier_dataset_df

,vendor_name,total_quantity_in,avg_shipped_receipt_diff_days,avg_shipped_bill_diff_days,using_goods_since_months,net_monetary_transaction,on_time_delivery_rate_percent,distinct_items_supplied,total_shipments
0,Sandhya Aqua Exports Pvt Ltd,19332352.0,52.61,48.65,74,73687968.08,1.96,26,550
1,Aquatica Frozen Foods Global Pvt Ltd,8280656.0,59.20,58.87,74,34932531.67,1.56,17,201
2,SANDHYA MARINES LTD,6354389.0,58.66,55.57,74,24970135.01,3.68,15,180
3,"NTSF Company, Inc",5199083.5,46.68,43.97,74,21575085.25,0.00,4,157
4,Suryamitra Exim LTD,4980220.0,61.06,61.06,73,19805890.89,2.37,16,111
5,Nekkanti Sea Foods Limited,4509672.5,64.00,62.04,71,16432500.75,3.03,11,131
6,Kalyan Aqua & Marine Exports India Pvt Ltd,2847108.0,69.49,69.49,72,9536256.95,4.65,9,73
7,"Zalo Fresh, Inc.",2714728.0,46.05,46.05,28,9561150.80,0.00,6,62
8,Geo Seafoods,2691852.0,59.36,59.36,74,9850450.70,4.94,4,76
9,Mangala Seafood,2444337.5,45.83,26.67,74,8679302.28,0.00,8,72


In [124]:
!pip install -q pymcdm

In [125]:
from pymcdm.methods import TOPSIS

# X = alternatives x criteria, weights = list, impacts = list
topsis = TOPSIS()
weights = [0.13, 0.13, 0.08, 0.09, 0.13, 0.22, 0.09, 0.13]

types = [1, -1, -1, 1, 1, 1, 1, 1]
X = supplier_dataset_df.iloc[:, 1:].values

rank = topsis(X, weights=weights, types=types)


In [126]:
supplier_dataset_df["topsis_score"] = rank * 100

In [127]:
supplier_dataset_df.sort_values(by='topsis_score', ascending=False)

,vendor_name,total_quantity_in,avg_shipped_receipt_diff_days,avg_shipped_bill_diff_days,using_goods_since_months,net_monetary_transaction,on_time_delivery_rate_percent,distinct_items_supplied,total_shipments,topsis_score
0,Sandhya Aqua Exports Pvt Ltd,19332352.0,52.61,48.65,74,73687968.08,1.96,26,550,55.952797
18,JAGADEESH MARINE EXPORTS,65880.0,54.50,54.50,57,263520.00,50.00,2,2,49.822089
1,Aquatica Frozen Foods Global Pvt Ltd,8280656.0,59.20,58.87,74,34932531.67,1.56,17,201,38.808186
2,SANDHYA MARINES LTD,6354389.0,58.66,55.57,74,24970135.01,3.68,15,180,36.788457
3,"NTSF Company, Inc",5199083.5,46.68,43.97,74,21575085.25,0.00,4,157,35.180193
9,Mangala Seafood,2444337.5,45.83,26.67,74,8679302.28,0.00,8,72,33.926552
4,Suryamitra Exim LTD,4980220.0,61.06,61.06,73,19805890.89,2.37,16,111,33.437737
5,Nekkanti Sea Foods Limited,4509672.5,64.00,62.04,71,16432500.75,3.03,11,131,31.336229
15,"IPSP, INC.",351400.0,25.20,25.20,12,1461544.00,0.00,2,4,31.244727
8,Geo Seafoods,2691852.0,59.36,59.36,74,9850450.70,4.94,4,76,30.151382
